# Inference Failover

Deploy two model-safe AI Gateway routes that exercise priority and weighted fallback across regional Azure OpenAI deployments while collecting focused LLM telemetry. See [README.md](README.md) for deployment details and prerequisites.

## What This Sample Does

- Deploys nine regional `Standard` Azure OpenAI deployments at one-thousand tokens-per-minute (`1,000 TPM`) for deliberate pressure testing.
- Exposes only `POST /inference/gpt-5-1/chat/completions` and `POST /inference/gpt-4-1-mini/chat/completions`.
- Routes each model through its own APIM backend pool, with managed identity authentication, TLS validation, circuit breaking, and buffered retries.
- Captures AOAI request, response, latency, and token telemetry in an Azure Monitor Workbook and in the verification charts below.

## Route Graph

Each operation targets one APIM backend pool. The pool selects a single member per request by priority (lowest first), and members that share a priority split traffic by weight. The backends never call one another; APIM picks the active backend and fails over to the next priority tier when one is unavailable.

### Why This Lab Uses `Standard` Deployments

Every target in this lab is a regional `Standard` pay-per-token deployment at `1` thousand TPM. This keeps the lab approachable, makes each regional target explicit, and makes concentrated traffic likely to produce observable `429` responses that exercise APIM circuit breaking and fallback. The `PTU` and `PAYG` labels describe the preference tiers being simulated; they do not change the real Azure OpenAI SKU or billing model.

For production, a better deployment type depends on model support, traffic shape, and data-residency requirements. For example, `gpt-4.1-mini` supports provisioned managed deployments. A regional `ProvisionedManaged` deployment reserves PTUs for predictable throughput and lower latency variance. Where the selected model and residency requirements permit, `GlobalStandard` can dynamically route across Azure datacenters and provide higher default quota, while `DataZoneStandard` can dynamically route within a US or EU data zone. This lab intentionally uses regional `Standard` deployments so every destination and APIM failover decision remain visible. See [deployment types](https://learn.microsoft.com/azure/foundry/foundry-models/concepts/deployment-types) and [model availability](https://learn.microsoft.com/azure/foundry/foundry-models/concepts/models-sold-directly-by-azure-region-availability) before choosing a production topology.

### Reading The Deployment Identifiers

The `a)` through `h)` prefixes are compact visual identifiers for concrete Azure OpenAI model deployments. In Azure, each prefix is part of the full deployment name, such as `a-gpt-5-1` or `c-gpt-4-1-mini`. The notebook uses these names in `X-Backend-URL` values and charts so a routed request can be traced back to the same graph node.

The letters are labels, not priorities. Read each letter together with its model name: for example, `d) gpt-5-1` and `d) gpt-4-1-mini` are different deployments. The graph-edge labels define the routing priority.

```mermaid
flowchart LR
  subgraph S5 [" gpt-5.1 (East US 2)"]
    C51["POST /inference/gpt-5-1/chat/completions"] --> P5{{"&nbsp;&nbsp;Backend pool: inference-gpt-5-1-pool&nbsp;&nbsp;"}}
    P5 -- "priority 1" --> A["`East US 2
                            PTU
                            a) gpt-5-1`"]
    P5 -- "priority 2" --> D["`West US 3
                            PTU
                            d) gpt-5-1`"]
    P5 -- "priority 3" --> B["`East US 2
                            PAYG
                            b) gpt-5-1`"]
    P5 -- "priority 4 &middot; weight 50" --> E["`West US 3 
                                                PAYG
                                                e) gpt-5-1`"]
    P5 -- "priority 4 &middot; weight 50" --> G["`South Central US
                                                PAYG
                                                g) gpt-5-1`"]
  end

  S5 ~~~ S4

  subgraph S4 [" gpt-4.1-mini (East US 2)"]
    C41["POST /inference/gpt-4-1-mini/chat/completions"] --> P4{{"&nbsp;&nbsp;Backend pool: inference-gpt-4-1-mini-pool&nbsp;&nbsp;"}}
    P4 -- "priority 1" --> C["`East US 2
                                PTU
                                c) gpt-4-1-mini`"]
    P4 -- "priority 2" --> F["`West US 3 
                                PTU
                                f) gpt-4-1-mini`"]
    P4 -- "priority 3" --> DP["`East US 2
                                PAYG
                                d) gpt-4-1-mini`"]
    P4 -- "priority 4" --> H["`South Central US
                                PAYG
                                h) gpt-4-1-mini`"]
  end
```


In [ ]:
import importlib
import sys
from pathlib import Path

# APIM Samples imports
import azure_resources as az
import utils
from apimtypes import APIM_SKU, INFRASTRUCTURE, Region
from console import print_ok

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location = Region.EAST_US_2
index = 110
apim_sku = APIM_SKU.BASICV2
deployment = INFRASTRUCTURE.SIMPLE_APIM
api_prefix = 'inference'
tags = ['inference-failover', 'ai-gateway', 'observability']
max_completion_tokens = 128                 # Per-request completion cap for the pressure scenarios

# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

sample_folder = 'inference-failover'
enable_event_hub_export = False             # Opt in to regional Event Hub streaming for APIM telemetry
rg_name = az.get_infra_rg_name(deployment, index)
supported_infras = [
    INFRASTRUCTURE.AFD_APIM_PE,
    INFRASTRUCTURE.APIM_ACA,
    INFRASTRUCTURE.APPGW_APIM,
    INFRASTRUCTURE.APPGW_APIM_PE,
    INFRASTRUCTURE.SIMPLE_APIM,
]
nb_helper = utils.NotebookHelper(
    sample_folder,
    rg_name,
    rg_location,
    deployment,
    supported_infras,
    index=index,
    apim_sku=apim_sku,
)
current_user, current_user_id, tenant_id, subscription_id = az.get_account_info()

sample_dir = str(Path(utils.determine_policy_path('inference_failover_helpers.py', sample_folder)).parent)
if sample_dir not in sys.path:
    sys.path.insert(0, sample_dir)
inference_failover_helpers = importlib.import_module('inference_failover_helpers')
utils.enable_module_autoreload('inference_failover_helpers')

print_ok('Notebook initialized')

## Deploy Inference Gateway Resources

Deploy the regional Azure OpenAI constellation, APIM backends and pools, retry fragment, LLM diagnostic wiring, two APIs, AOAI-only workbook, and optional regional Event Hub telemetry stream into the selected infrastructure resource group.

In [ ]:
# APIM Samples imports
from apimtypes import API, APIOperation, HTTP_VERB
from console import print_ok, print_val

policy_template = utils.read_policy_xml('inference-api-policy.xml', sample_name=sample_folder)


def build_inference_policy(backend_pool_id: str, retry_count: int) -> str:
    """Build one model-safe inference API policy from the sample template."""
    return policy_template.replace('BACKEND_POOL_ID', backend_pool_id).replace('RETRY_COUNT', str(retry_count))


chat_operation = APIOperation(
    'chat-completions',
    'Chat Completions',
    '/chat/completions',
    HTTP_VERB.POST,
    'Send an inference request through a model-safe backend pool.',
)
gpt_5_1_api = API(
    'inference-gpt-5-1',
    'Inference gpt-5.1',
    f'{api_prefix}/gpt-5-1',
    'Priority and weighted failover for gpt-5.1 deployments.',
    policyXml=build_inference_policy('inference-gpt-5-1-pool', 4),
    operations=[chat_operation],
    tags=tags,
    enableLlmLogging=True,
)
gpt_4_1_mini_api = API(
    'inference-gpt-4-1-mini',
    'Inference gpt-4.1-mini',
    f'{api_prefix}/gpt-4-1-mini',
    'Priority failover for gpt-4.1-mini deployments.',
    policyXml=build_inference_policy('inference-gpt-4-1-mini-pool', 3),
    operations=[chat_operation],
    tags=tags,
    enableLlmLogging=True,
)
apis = [gpt_5_1_api, gpt_4_1_mini_api]

bicep_parameters = {
    'location': {'value': rg_location},
    'index': {'value': index},
    'apis': {'value': [api.to_dict() for api in apis]},
    'enableEventHubExport': {'value': enable_event_hub_export},
}
output = nb_helper.deploy_sample(bicep_parameters)
deployment_context = nb_helper.get_deployment_context(output)
apim_name = deployment_context.apim_name
apim_gateway_url = deployment_context.apim_gateway_url
api_outputs = deployment_context.apis
log_analytics_name = output.get('logAnalyticsWorkspaceName', 'Log Analytics Workspace Name')
log_analytics_id = output.get('logAnalyticsWorkspaceId', 'Log Analytics Workspace ID')
workbook_name = output.get('workbookName', 'Workbook Name')
workbook_id = output.get('workbookId', 'Workbook ID')
event_hub_export_enabled = output.get('eventHubExportEnabled', 'Event Hub export enabled')
event_hub_namespace_id = output.get('eventHubNamespaceId', 'Event Hubs namespace ID')
event_hub_namespace_name = output.get('eventHubNamespaceName', 'Event Hubs namespace name')
event_hub_id = output.get('eventHubId', 'Event Hub ID')
event_hub_name = output.get('eventHubName', 'Event Hub name')
event_hub_consumer_group_name = output.get('eventHubConsumerGroupName', 'Event Hub consumer group name')
event_hub_location = output.get('eventHubLocation', 'Event Hubs namespace location')
api_keys = {api_output['name']: api_output['subscriptionPrimaryKey'] for api_output in api_outputs}
print_val('Workbook', workbook_name)
print_val('Event Hub export enabled', event_hub_export_enabled)
if event_hub_export_enabled:
    print_val('Event Hubs namespace', event_hub_namespace_name)
    print_val('Event Hub', event_hub_name)
    print_val('Event Hub consumer group', event_hub_consumer_group_name)
    print_val('Event Hubs location', event_hub_location)
print_ok('Inference failover deployment completed successfully')

## Deterministic Gateway Contract Probes

Verify the status codes that the deployed gateway can reproduce safely and consistently before generating load:

| Probe | Trigger | Expected result |
| --- | --- | --- |
| Healthy request | Valid chat-completions payload and subscription key | `200` with `X-Backend-Retry: 0` |
| Client error | Truncated JSON sent to Azure OpenAI | `400` with `X-Backend-Retry: 0` |
| Authentication error | Omit the APIM subscription key | Gateway `401` before the inference policy runs |
| Configuration error | Call an unknown APIM operation | Gateway `404` before the inference policy runs |

The `401` and gateway-level `404` intentionally have no `X-Backend-Retry` header because API matching or subscription validation rejects them before the inference policy executes. See the README's **Failure Test Coverage** section for controlled `403`, both `409` branches, deterministic `429`, infrastructure statuses, DNS/TLS failures, timeouts, and true caller disconnects.

In [ ]:
# APIM Samples imports
from apimtesting import ApimTesting
from console import print_error, print_ok

if 'api_keys' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

probe_api_name = 'inference-gpt-5-1'
probe_subscription_key = api_keys.get(probe_api_name)
if not probe_subscription_key:
    print_error(f'Missing API subscription key for {probe_api_name}')
    raise SystemExit(1)

probe_endpoint_url, probe_request_headers, probe_allow_insecure_tls = utils.get_endpoint(deployment, rg_name, apim_gateway_url)
probe_route = f'{probe_endpoint_url}/{api_prefix}/gpt-5-1/chat/completions'
unknown_operation_route = f'{probe_endpoint_url}/{api_prefix}/gpt-5-1/not-an-operation'
probe_payload = {
    'messages': [{'role': 'user', 'content': 'Return the word ready.'}],
    'max_completion_tokens': 16,
}
contract_tests = ApimTesting('Inference Gateway Contract Probes', sample_folder, nb_helper.deployment)

with inference_failover_helpers.InferenceTrafficRunner(
    probe_endpoint_url,
    probe_request_headers,
    probe_allow_insecure_tls,
) as probe_runner:
    probe_results = probe_runner.run_contract_probes(
        probe_route,
        unknown_operation_route,
        probe_subscription_key,
        probe_payload,
    )

contract_tests.verify(probe_results.success.status_code, 200, label='Valid inference request returns HTTP 200')
contract_tests.verify(probe_results.success.headers.get('X-Backend-Retry'), '0', label='Healthy response performs no backend retry')
contract_tests.verify(probe_results.malformed.status_code, 400, label='Malformed JSON returns backend HTTP 400')
contract_tests.verify(probe_results.malformed.headers.get('X-Backend-Retry'), '0', label='HTTP 400 is not retried')
contract_tests.verify(probe_results.missing_key.status_code, 401, label='Missing APIM subscription key returns HTTP 401')
contract_tests.verify(
    'X-Backend-Retry' in probe_results.missing_key.headers,
    False,
    label='Gateway HTTP 401 occurs before the inference policy',
)
contract_tests.verify(probe_results.unknown_operation.status_code, 404, label='Unknown APIM operation returns HTTP 404')
contract_tests.verify(
    'X-Backend-Retry' in probe_results.unknown_operation.headers,
    False,
    label='Gateway HTTP 404 occurs before the inference policy',
)

print_ok('Deterministic gateway contract probes completed')

## 🧪 Inference Failover Test Matrix

Run a methodical series of scenarios that exercise the model-safe backend pools under increasing pressure, mirroring the structure of the load-balancing sample. Each scenario sends real Azure OpenAI requests through a single shared session, captures the served backend from the `X-Backend-URL` response header, and records any HTTP `4xx` or `5xx` response so later requests continue to run.

| # | Scenario | API | Runs | Sleep | Behaviour verified |
| --- | --- | --- | --- | --- | --- |
| 1 | Baseline warm path | gpt-5.1 / gpt-4.1-mini | 3 / 6 | none | Fresh priority-1 backend serves small control requests with `200` |
| 2 | Sustained pressure | gpt-5.1 | 45 | none | Low-TPM exhaustion forces throttling and priority failover |
| 3 | Sustained pressure | gpt-4.1-mini | 15 | none | Independent pool exhausts and fails over on its own |
| 4 | Paced recovery | gpt-5.1 | 30 | 1.5 s | Inter-request spacing lets circuit breakers recover and lifts the success rate |
| 5 | Terminal exhaustion | gpt-5.1 | 35 | none | A hard burst drains every tier and can return the generic `503` |

> These are real Azure OpenAI calls (`134` requests) that consume tokens. The deployments are intentionally provisioned at minimum capacity so that pressure is observable quickly; this is an observation aid, not production sizing guidance.

In [ ]:
# APIM Samples imports
from apimtesting import ApimTesting
from console import print_error, print_info, print_message, print_ok

if 'api_keys' not in locals():
    print_error('Please run the deployment cell first')
    raise SystemExit(1)

required_api_names = ['inference-gpt-5-1', 'inference-gpt-4-1-mini']
missing_api_keys = [api_name for api_name in required_api_names if not api_keys.get(api_name)]
if missing_api_keys:
    print_error(f'Missing API subscription keys for: {", ".join(missing_api_keys)}')
    print_error('Please rerun the deployment cell to refresh the API subscription outputs')
    raise SystemExit(1)

endpoint_url, request_headers, allow_insecure_tls = utils.get_endpoint(deployment, rg_name, apim_gateway_url)

# URL markers are listed in deployed pool-priority order so chart colors track failover tiers.
gpt_5_1_backend_url_index = {
    '/openai/deployments/a-gpt-5-1': 0,   # P1 in-region PTU
    '/openai/deployments/d-gpt-5-1': 1,   # P2 out-of-region PTU
    '/openai/deployments/b-gpt-5-1': 2,   # P3 in-region PAYG
    '/openai/deployments/e-gpt-5-1': 3,   # P4 out-of-region PAYG (westus3)
    '/openai/deployments/g-gpt-5-1': 4,   # P4 out-of-region PAYG (southcentralus)
}
gpt_4_1_mini_backend_url_index = {
    '/openai/deployments/c-gpt-4-1-mini': 0,   # P1 in-region PTU
    '/openai/deployments/f-gpt-4-1-mini': 1,   # P2 out-of-region PTU
    '/openai/deployments/d-gpt-4-1-mini': 2,   # P3 in-region PAYG
    '/openai/deployments/h-gpt-4-1-mini': 3,   # P4 out-of-region PAYG
}

gpt_5_1_route = f'{api_prefix}/gpt-5-1/chat/completions'
gpt_4_1_mini_route = f'{api_prefix}/gpt-4-1-mini/chat/completions'
gpt_5_1_baseline_runs = 3
gpt_4_1_mini_baseline_runs = 6

small_payload = {
    'messages': [{'role': 'user', 'content': 'Return the word ready.'}],
    'max_completion_tokens': 16,
}
pressure_payload = {
    'messages': [{'role': 'user', 'content': 'Summarize failover readiness in three bullets. ' * 20}],
    'max_completion_tokens': max_completion_tokens,
}

scenarios = [
    inference_failover_helpers.InferenceScenario(
        '1/5: Baseline warm path - gpt-5.1',
        gpt_5_1_route,
        api_keys['inference-gpt-5-1'],
        small_payload,
        gpt_5_1_baseline_runs,
        gpt_5_1_backend_url_index,
    ),
    inference_failover_helpers.InferenceScenario(
        '1/5: Baseline warm path - gpt-4.1-mini',
        gpt_4_1_mini_route,
        api_keys['inference-gpt-4-1-mini'],
        small_payload,
        gpt_4_1_mini_baseline_runs,
        gpt_4_1_mini_backend_url_index,
    ),
    inference_failover_helpers.InferenceScenario(
        '2/5: Sustained pressure - gpt-5.1 (no spacing)',
        gpt_5_1_route,
        api_keys['inference-gpt-5-1'],
        pressure_payload,
        45,
        gpt_5_1_backend_url_index,
    ),
    inference_failover_helpers.InferenceScenario(
        '3/5: Sustained pressure - gpt-4.1-mini (no spacing)',
        gpt_4_1_mini_route,
        api_keys['inference-gpt-4-1-mini'],
        pressure_payload,
        15,
        gpt_4_1_mini_backend_url_index,
    ),
    inference_failover_helpers.InferenceScenario(
        '4/5: Paced recovery - gpt-5.1 (1.5s spacing)',
        gpt_5_1_route,
        api_keys['inference-gpt-5-1'],
        pressure_payload,
        30,
        gpt_5_1_backend_url_index,
        sleep_ms=1500,
    ),
    inference_failover_helpers.InferenceScenario(
        '5/5: Terminal exhaustion - gpt-5.1 (hard burst)',
        gpt_5_1_route,
        api_keys['inference-gpt-5-1'],
        pressure_payload,
        35,
        gpt_5_1_backend_url_index,
    ),
]

tests = ApimTesting('Inference Failover Scenario Tests', sample_folder, nb_helper.deployment)
scenario_results = []
with inference_failover_helpers.InferenceTrafficRunner(endpoint_url, request_headers, allow_insecure_tls) as traffic_runner:
    for scenario_index, scenario in enumerate(scenarios):
        if scenario_index in (2, 3, 4, 5):
            traffic_runner.pause(2)
        scenario_results.append(traffic_runner.run_scenario(scenario))

(
    scenario1_gpt_5_1,
    scenario1_gpt_4_1_mini,
    scenario2_gpt_5_1,
    scenario3_gpt_4_1_mini,
    scenario4_gpt_5_1,
    scenario5_gpt_5_1,
) = scenario_results

tests.verify(len(scenario1_gpt_5_1), gpt_5_1_baseline_runs)
tests.verify(len(scenario1_gpt_4_1_mini), gpt_4_1_mini_baseline_runs)
tests.verify(scenario1_gpt_5_1[0]['status_code'], 200, label='gpt-5.1 baseline first request succeeded')
tests.verify(scenario1_gpt_4_1_mini[0]['status_code'], 200, label='gpt-4.1-mini baseline first request succeeded')

for scenario_name, results, expected_runs in [
    ('gpt-5.1 sustained pressure', scenario2_gpt_5_1, 45),
    ('gpt-4.1-mini sustained pressure', scenario3_gpt_4_1_mini, 15),
    ('gpt-5.1 paced recovery', scenario4_gpt_5_1, 30),
    ('gpt-5.1 terminal burst', scenario5_gpt_5_1, 35),
]:
    summary = inference_failover_helpers.summarize_scenario(results)
    tests.verify(len(results), expected_runs)
    tests.verify(
        summary.successes + summary.client_errors + summary.server_errors == len(results),
        True,
        label=f'{scenario_name} captured every request',
    )
    if scenario_name != 'gpt-5.1 terminal burst':
        tests.verify(summary.successes > 0, True, label=f'{scenario_name} served some requests')
    if summary.client_errors + summary.server_errors:
        print_info(
            f'{scenario_name} observed {summary.client_errors} 4xx and {summary.server_errors} 5xx responses: '
            f'{summary.status_code_counts}'
        )
    else:
        print_info(f'{scenario_name} was fully absorbed by failover (no 4xx/5xx)')

print_message('Scenario success / 4xx / 5xx summary', blank_above=True)
for scenario_name, results in [
    ('1 baseline gpt-5.1', scenario1_gpt_5_1),
    ('1 baseline gpt-4.1-mini', scenario1_gpt_4_1_mini),
    ('2 sustained gpt-5.1', scenario2_gpt_5_1),
    ('3 sustained gpt-4.1-mini', scenario3_gpt_4_1_mini),
    ('4 paced gpt-5.1', scenario4_gpt_5_1),
    ('5 terminal gpt-5.1', scenario5_gpt_5_1),
]:
    summary = inference_failover_helpers.summarize_scenario(results)
    print_info(
        f'Scenario {scenario_name}: {summary.successes} ok, {summary.client_errors} 4xx, {summary.server_errors} 5xx; '
        f'statuses: {summary.status_code_counts}'
    )
    print_info(f'Observed X-Backend-Retry values: {summary.retry_counts}')
    print_info(f'Observed X-Backend-URL values: {summary.served_backend_urls}')

tests.print_summary()
if tests.tests_failed:
    raise SystemExit(1)
print_ok('All inference failover scenarios completed')

## 📊 Analyze Inference Failover Results

Plot the per-run response time for each scenario in the same style as the load-balancing sample. Bars are colored by the backend that served the request, derived from the infrastructure-level `X-Backend-URL` response header. Each chart title identifies the APIM source region selected during deployment, and each legend entry identifies the destination priority tier, capacity profile, and region.

#### Why `X-Backend-URL` is the routing indicator

The infrastructure's global APIM policy returns the final resolved request URL after the backend pool selects a member. The inference harness stores that full URL on every result and derives the chart label from its deployment path marker. This keeps the chart evidence tied to the actual Azure OpenAI destination rather than a separately maintained backend label.

#### Why the baseline bars should use one backend

The baseline is a control, not a distribution test. Before pressure is applied, APIM should keep selecting the healthy priority-1 backend for all three gpt-5.1 requests and all six gpt-4.1-mini requests. That repetition confirms each preferred route is stable, warms the connection and model path, and establishes a latency reference. Compare scenarios 2 through 5 against that control to see when pressure or recovery moves traffic into lower-priority tiers.

#### Reading the charts

- Each chart title shows the APIM source region. Use it with the legend's destination regions to correlate source, destination, and latency.
- Each successful bar's legend label names its priority tier, simulated capacity profile (`PTU` or `PAYG`), and Azure OpenAI destination region.
- A run colored red is a non-`200` response. The harness records any HTTP `4xx` or `5xx` response and continues the scenario so the full pressure window remains visible.
- The first request in a cold scenario is often slower (TLS warm-up plus first-token latency); the mean line ignores the slowest 5% so it reflects steady-state behaviour.
- As pressure builds, the colors can walk from priority 1 into lower-priority tiers as the preferred backend trips and traffic fails over.

**Backend URL marker legend**

| gpt-5.1 chart label                              | gpt-5.1 pool URL marker                  | gpt-4.1-mini chart label                   | gpt-4.1-mini pool URL marker                  |
| ------------------------------------------------ | ---------------------------------------- | ------------------------------------------ | --------------------------------------------- |
| `Priority 1: PTU (East US 2)`                    | `/openai/deployments/a-gpt-5-1`          | `Priority 1: PTU (East US 2)`              | `/openai/deployments/c-gpt-4-1-mini`          |
| `Priority 2: PTU (West US 3)`                    | `/openai/deployments/d-gpt-5-1`          | `Priority 2: PTU (West US 3)`              | `/openai/deployments/f-gpt-4-1-mini`          |
| `Priority 3: PAYG (East US 2)`                   | `/openai/deployments/b-gpt-5-1`          | `Priority 3: PAYG (East US 2)`             | `/openai/deployments/d-gpt-4-1-mini`          |
| `Priority 4: PAYG (West US 3, weight 50)`        | `/openai/deployments/e-gpt-5-1`          | `Priority 4: PAYG (South Central US)`       | `/openai/deployments/h-gpt-4-1-mini`          |
| `Priority 4: PAYG (South Central US, weight 50)` | `/openai/deployments/g-gpt-5-1`          | -                                          | -                                             |

> If a successful bar uses a fallback `Backend index 99` legend entry, inspect its captured `X-Backend-URL` value. The resolved URL did not contain one of the expected deployment markers, so the per-run backend labeling needs to be updated for the deployed topology.


In [ ]:
import importlib

from IPython.display import Markdown, display

# APIM Samples imports
import charts
from console import print_error

charts = importlib.reload(charts)

if 'scenario5_gpt_5_1' not in locals():
    print_error('Please run the scenario cell first')
    raise SystemExit(1)


def display_scenario_context(title: str, explanation: str) -> None:
    """Render a short explanation immediately before scenario charts."""
    display(Markdown(f'#### {title}\n\n{explanation}'))


format_backend_url_counts = inference_failover_helpers.format_backend_url_counts
apim_source_region = rg_location.name.replace('_', ' ').title().replace(' Us', ' US').replace(' Uk', ' UK')
gpt_5_1_backend_labels = {
    0: 'Priority 1 / Weight 100: PTU (East US 2)',
    1: 'Priority 2 / Weight 100: PTU (West US 3)',
    2: 'Priority 3 / Weight 100: PAYG (East US 2)',
    3: 'Priority 4 / Weight 50: PAYG (West US 3)',
    4: 'Priority 4 / Weight 50: PAYG (South Central US)',
}
gpt_4_1_mini_backend_labels = {
    0: 'Priority 1 / Weight 100: PTU (East US 2)',
    1: 'Priority 2 / Weight 100: PTU (West US 3)',
    2: 'Priority 3 / Weight 100: PAYG (East US 2)',
    3: 'Priority 4 / Weight 100: PAYG (South Central US)',
}


display_scenario_context(
    'Baseline Warm Path',
    (
        'These control graphs show end-to-end response time and the resolved backend for small requests before deliberate pressure begins. '
        'They confirm that each independent pool starts from a healthy priority-1 route and provide a latency baseline for comparison.'
    ),
)

# 1a) Baseline warm path - gpt-5.1
charts.BarChart(
    api_results = scenario1_gpt_5_1,
    title = f'Baseline Warm Path (gpt-5.1) - APIM source: {apim_source_region}',
    x_label = 'Run #',
    y_label = 'Response Time (ms)',
    fig_text = (
        f'The chart shows a total of {len(scenario1_gpt_5_1)} small requests against the gpt-5.1 backend pool.\n'
        'A fresh pool should start on its priority-1 in-region backend.\n'
        f'Observed X-Backend-URL values:\n{format_backend_url_counts(scenario1_gpt_5_1)}\n'
        'The average response time is calculated excluding statistical outliers above the 95th percentile '
        '(the first request usually takes longer).'
    ),
    backend_labels = gpt_5_1_backend_labels,
).plot()

# 1b) Baseline warm path - gpt-4.1-mini
charts.BarChart(
    api_results = scenario1_gpt_4_1_mini,
    title = f'Baseline Warm Path (gpt-4.1-mini) - APIM source: {apim_source_region}',
    x_label = 'Run #',
    y_label = 'Response Time (ms)',
    fig_text = (
        f'The chart shows a total of {len(scenario1_gpt_4_1_mini)} small requests against the gpt-4.1-mini backend pool.\n'
        'A fresh pool should start on its priority-1 in-region backend.\n'
        f'Observed X-Backend-URL values:\n{format_backend_url_counts(scenario1_gpt_4_1_mini)}\n'
        'The average response time is calculated excluding statistical outliers above the 95th percentile '
        '(the first request usually takes longer).'
    ),
    backend_labels = gpt_4_1_mini_backend_labels,
).plot()

display_scenario_context(
    'Sustained Pressure: gpt-5.1',
    (
        'This graph shows how the gpt-5.1 pool behaves when larger requests arrive without spacing. '
        'Reveals whether circuit breakers redistribute traffic through the priority tiers and whether the weighted terminal tier absorbs the burst.'
    ),
)

# 2) Sustained pressure - gpt-5.1
charts.BarChart(
    api_results = scenario2_gpt_5_1,
    title = f'Sustained Pressure (gpt-5.1) - APIM source: {apim_source_region}',
    x_label = 'Run #',
    y_label = 'Response Time (ms)',
    fig_text = (
        f'The chart shows a total of {len(scenario2_gpt_5_1)} requests with no spacing against the gpt-5.1 backend pool.\n'
        'Low-capacity deployments can trip under pressure, moving traffic through the priority tiers and into the equally-weighted terminal tier.\n'
        f'Observed X-Backend-URL values:\n{format_backend_url_counts(scenario2_gpt_5_1)}\n'
        'The average response time is calculated excluding statistical outliers above the 95th percentile.'
    ),
    backend_labels = gpt_5_1_backend_labels,
).plot()

display_scenario_context(
    'Sustained Pressure: gpt-4.1-mini',
    (
        'This graph applies the same unpaced load to the independent gpt-4.1-mini pool. '
        'It confirms that failover remains model-safe: pressure on this route can only move traffic among compatible gpt-4.1-mini deployments.'
    ),
)

# 3) Sustained pressure - gpt-4.1-mini
charts.BarChart(
    api_results = scenario3_gpt_4_1_mini,
    title = f'Sustained Pressure (gpt-4.1-mini) - APIM source: {apim_source_region}',
    x_label = 'Run #',
    y_label = 'Response Time (ms)',
    fig_text = (
        f'The chart shows a total of {len(scenario3_gpt_4_1_mini)} requests with no spacing against the independent gpt-4.1-mini pool.\n'
        'The model-specific pool keeps fallback on compatible deployments while priority tiers respond to pressure.\n'
        f'Observed X-Backend-URL values:\n{format_backend_url_counts(scenario3_gpt_4_1_mini)}\n'
        'The average response time is calculated excluding statistical outliers above the 95th percentile.'
    ),
    backend_labels = gpt_4_1_mini_backend_labels,
).plot()

display_scenario_context(
    'Paced Recovery',
    (
        'This graph repeats the gpt-5.1 pressure workload with a short pause between requests. '
        'Comparing it with sustained pressure shows whether circuit-breaker recovery improves availability and returns traffic to preferred tiers.'
    ),
)

# 4) Paced recovery - gpt-5.1
charts.BarChart(
    api_results = scenario4_gpt_5_1,
    title = f'Paced Recovery (gpt-5.1, 1.5s sleep) - APIM source: {apim_source_region}',
    x_label = 'Run #',
    y_label = 'Response Time (ms)',
    fig_text = (
        f'The chart shows a total of {len(scenario4_gpt_5_1)} requests with a 1.5-second sleep after each request.\n'
        'Inter-request spacing gives tripped backends time to recover while the same priority and weighted pool continues serving traffic.\n'
        f'Observed X-Backend-URL values:\n{format_backend_url_counts(scenario4_gpt_5_1)}\n'
        'The average response time is calculated excluding statistical outliers above the 95th percentile.'
    ),
    backend_labels = gpt_5_1_backend_labels,
).plot()

display_scenario_context(
    'Terminal Exhaustion',
    (
        'This graph probes the deepest fallback behavior with a hard gpt-5.1 burst. '
        'Demonstrates whether the complete pool absorbs the load and, when capacity is exhausted, whether callers receive a clean terminal response.'
    ),
)

# 5) Terminal exhaustion - gpt-5.1
charts.BarChart(
    api_results = scenario5_gpt_5_1,
    title = f'Terminal Exhaustion (gpt-5.1, hard burst) - APIM source: {apim_source_region}',
    x_label = 'Run #',
    y_label = 'Response Time (ms)',
    fig_text = (
        f'The chart shows a total of {len(scenario5_gpt_5_1)} hard-burst requests against the gpt-5.1 backend pool.\n'
        'A deep pool may absorb the burst; if all eligible tiers drain, red bars show terminal non-200 responses.\n'
        f'Observed X-Backend-URL values:\n{format_backend_url_counts(scenario5_gpt_5_1)}\n'
        'The average response time is calculated excluding statistical outliers above the 95th percentile.'
    ),
    backend_labels = gpt_5_1_backend_labels,
).plot()


## [Verify] Wait For AI Gateway Telemetry

Poll the existing Log Analytics workspace for gateway rows and token-bearing LLM diagnostic rows created by the traffic block. Ingestion normally takes several minutes.

In [ ]:
from pathlib import Path

# APIM Samples imports
from console import print_error, print_ok, print_warning

if 'log_analytics_id' not in locals():
    print_error('Please run the deployment and traffic cells first')
    raise SystemExit(1)

api_ids_binding = "let apiIds = dynamic(['inference-gpt-5-1', 'inference-gpt-4-1-mini']);\n"
time_binding = 'let timeWindow = 30m;\n'
queries_path = Path(utils.get_project_root()) / 'samples' / sample_folder / 'queries'
verify_template = (queries_path / 'verify-llm-ingestion.kql').read_text(encoding='utf-8')
verification_query = time_binding + api_ids_binding + verify_template
telemetry_found, telemetry_result, telemetry_rows = nb_helper.wait_for_kql(
    log_analytics_id,
    verification_query,
    api_path='/api/query',
    api_version='2020-08-01',
    progress_label='AI telemetry rows',
)

if telemetry_found:
    request_rows, successful_requests, unavailable_responses, token_rows, total_tokens = telemetry_rows[0]
    print_ok(f'Telemetry ready: {request_rows} requests, {token_rows} token-bearing requests, {total_tokens} tokens')
    if unavailable_responses:
        print_ok(f'Observed {unavailable_responses} terminal 503 responses after fallback exhaustion')
elif telemetry_result is not None and telemetry_result.success:
    print_warning('Token-bearing telemetry has not arrived yet; wait a few minutes and rerun this cell')

## [Verify] Plot Backend Distribution And Token Volume

Query the same AOAI-only data surfaced in the deployed workbook and render local graphs for route distribution and token consumption.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown as IPythonMarkdown
from IPython.display import display as ipython_display

# APIM Samples imports
from console import print_info, print_warning

queries_path = Path(utils.get_project_root()) / 'samples' / sample_folder / 'queries'
distribution_template = (queries_path / 'backend-distribution.kql').read_text(encoding='utf-8')
token_template = (queries_path / 'token-throughput.kql').read_text(encoding='utf-8')
bindings = "let timeWindow = 1h;\nlet apiIds = dynamic(['inference-gpt-5-1', 'inference-gpt-4-1-mini']);\n"
distribution_found, _, distribution_rows = nb_helper.wait_for_kql(
    log_analytics_id,
    bindings + distribution_template,
    schedule=[0],
    progress_label='distribution rows',
)
tokens_found, _, token_rows = nb_helper.wait_for_kql(
    log_analytics_id,
    bindings + token_template,
    schedule=[0],
    progress_label='token rows',
)

if distribution_found:
    ipython_display(IPythonMarkdown(
        '#### Gateway-Recorded Backend Distribution\n\n'
        'This table and graph summarize which regional backend handled each gateway request. '
        'The distribution shows how traffic moved across model-safe pool members as pressure and recovery scenarios '
        'exercised the failover tiers.'
    ))
    distribution_frame = pd.DataFrame(
        distribution_rows,
        columns=['API', 'Backend URL', 'Requests', 'Successes', 'Throttled', 'Failures', 'AverageBackendMs', 'SuccessRate'],
    )
    distribution_frame = inference_failover_helpers.with_backend_identifier(distribution_frame)
    distribution_frame = inference_failover_helpers.format_gateway_distribution(distribution_frame)
    display(distribution_frame)
    distribution_frame.pivot(index='API', columns='Backend', values='Requests').fillna(0).plot(
        kind='bar',
        figsize=(12, 5),
        title='Gateway-recorded request distribution by backend',
    )
    plt.xlabel('API')
    plt.ylabel('Requests')
    plt.tight_layout()
    plt.show()
else:
    print_warning('No gateway distribution rows returned yet')
    cached_distribution_frame = locals().get('distribution_frame')
    if cached_distribution_frame is not None:
        distribution_frame = inference_failover_helpers.with_backend_identifier(cached_distribution_frame)
        distribution_frame = inference_failover_helpers.format_gateway_distribution(distribution_frame)

if tokens_found:
    ipython_display(IPythonMarkdown(
        '#### LLM Token Volume\n\n'
        'This table and graph summarize total Azure OpenAI tokens observed for each inference API. '
        'Token volume helps connect the routing exercise to model consumption and confirms that LLM diagnostics '
        'captured successful inference traffic.'
    ))
    token_frame = pd.DataFrame(
        token_rows,
        columns=['API', 'Backend URL', 'Model', 'Requests', 'PromptTokens', 'CompletionTokens', 'TotalTokens'],
    )
    token_frame = inference_failover_helpers.with_backend_identifier(token_frame)
    display(token_frame)
    token_frame.groupby('API')['TotalTokens'].sum().plot(
        kind='bar',
        figsize=(8, 4),
        title='LLM token volume by inference API',
    )
    plt.xlabel('Inference API')
    plt.ylabel('Total tokens')
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print_info('No token rows returned yet; successful inference rows may still be ingesting')
    cached_token_frame = locals().get('token_frame')
    if cached_token_frame is not None:
        token_frame = inference_failover_helpers.with_backend_identifier(cached_token_frame)

## Generate Local HTML Report

Create a self-contained local HTML report with test totals, scenario outcomes, response-time graphs, telemetry tables, and telemetry graphs. The closing cell prints a link to the generated file.

In [ ]:
import importlib
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt

# APIM Samples imports
import charts
import htmlreport
from console import print_ok, print_warning

importlib.reload(htmlreport)

if 'gpt_5_1_backend_labels' not in locals():
    print_warning('Please run the scenario chart cell first')
    raise SystemExit(1)


def add_scenario_figure(
    report: htmlreport.HtmlReport,
    title: str,
    description: str,
    api_results: list[dict],
    backend_labels: dict[int, str],
) -> None:
    """Render and embed one scenario response-time chart."""
    figure = charts.BarChart(
        api_results=api_results,
        title=f'{title} - APIM source: {apim_source_region}',
        x_label='Run #',
        y_label='Response Time (ms)',
        fig_text=description,
        backend_labels=backend_labels,
    ).render()
    report.add_figure(title, figure, description)
    plt.close(figure)


portal_base = f'https://portal.azure.com/#@{tenant_id}/resource'
apim_id = f'/subscriptions/{subscription_id}/resourceGroups/{rg_name}/providers/Microsoft.ApiManagement/service/{apim_name}'
report = htmlreport.HtmlReport(
    'Inference Failover Run Report',
    f'APIM source region: {apim_source_region} | Deployment: {nb_helper.deployment.name} | Resource group: {rg_name}',
)
success_rate = (tests.tests_passed / tests.total_tests * 100) if tests.total_tests else 0
report.add_metrics(
    'Scenario Test Summary',
    {
        'Result': 'Passed' if not tests.tests_failed and tests.total_tests else 'Needs review',
        'Tests': tests.total_tests,
        'Passed': tests.tests_passed,
        'Failed': tests.tests_failed,
        'Success rate': f'{success_rate:.1f}%',
    },
    highlight_success=False,
)
report.add_info_callout(
    'Lab Capacity Is Intentionally Low',
    'Each regional Azure OpenAI deployment is intentionally configured at 1,000 TPM so that concentrated requests trigger observable failover. '
    'This is a learning configuration, not production sizing guidance. With appropriately sized production capacity, '
    'caller-visible success rates should be substantially higher and, normally, close to perfect.',
)
if tests.errors:
    report.add_table('Assertion Failures', ['Failure'], [[error] for error in tests.errors])

scenario_groups = [
    (
        'gpt-5.1',
        [
            (
                'A-1',
                'Baseline Warm Path',
                'Small control requests against the gpt-5.1 pool before deliberate pressure begins.',
                scenario1_gpt_5_1,
                gpt_5_1_backend_url_index,
                gpt_5_1_backend_labels,
            ),
            (
                'A-2',
                'Sustained Pressure',
                'Larger requests without spacing exercise gpt-5.1 failover tiers.',
                scenario2_gpt_5_1,
                gpt_5_1_backend_url_index,
                gpt_5_1_backend_labels,
            ),
            (
                'A-3',
                'Paced Recovery',
                'Spaced requests show recovery behavior after sustained pressure.',
                scenario4_gpt_5_1,
                gpt_5_1_backend_url_index,
                gpt_5_1_backend_labels,
            ),
            (
                'A-4',
                'Terminal Exhaustion',
                'A hard burst probes deep fallback and terminal responses.',
                scenario5_gpt_5_1,
                gpt_5_1_backend_url_index,
                gpt_5_1_backend_labels,
            ),
        ],
    ),
    (
        'gpt-4.1-mini',
        [
            (
                'B-1',
                'Baseline Warm Path',
                'Small control requests against the independent gpt-4.1-mini pool.',
                scenario1_gpt_4_1_mini,
                gpt_4_1_mini_backend_url_index,
                gpt_4_1_mini_backend_labels,
            ),
            (
                'B-2',
                'Sustained Pressure',
                'Unpaced load confirms that gpt-4.1-mini fallback remains model-safe.',
                scenario3_gpt_4_1_mini,
                gpt_4_1_mini_backend_url_index,
                gpt_4_1_mini_backend_labels,
            ),
        ],
    ),
]
scenario_definitions = [scenario_definition for _, model_scenarios in scenario_groups for scenario_definition in model_scenarios]
scenario_request_count = sum(len(api_results) for _, _, _, api_results, _, _ in scenario_definitions)
all_scenario_requests_succeeded = bool(scenario_request_count) and all(
    result['status_code'] == 200
    for _, _, _, api_results, _, _ in scenario_definitions
    for result in api_results
)
if all_scenario_requests_succeeded:
    report.add_success_callout(
        'All scenario requests returned HTTP 200',
        f'All {scenario_request_count} controlled inference requests completed successfully. '
        'APIM served the full run without a caller-visible error.',
    )
for model_name, model_scenarios in scenario_groups:
    report.add_table(
        htmlreport.HtmlText(f'Scenario Outcomes: {model_name}', bold_tokens=(model_name,)),
        ['', 'Test #', 'Scenario', 'Requests', 'HTTP 200', 'Other', 'APIM retries', 'Priority / weight mix', 'What the data says'],
        [
            inference_failover_helpers.build_scenario_report_row(test_id, title, api_results, backend_url_index, backend_labels)
            for test_id, title, _, api_results, backend_url_index, backend_labels in model_scenarios
        ],
        htmlreport.HtmlText(
            'The green checkmark indicates that every request returned HTTP 200 from the caller perspective, '
            'including successes after APIM retries. '
            'The amber warning triangle indicates that one or more requests returned a caller-visible non-200 response. '
            'APIM retries lists only non-zero X-Backend-Retry values absorbed by APIM after the initial backend attempt. '
            'Priority / weight mix aggregates concrete Azure OpenAI destinations into APIM routing tiers and weights. '
            'The interpretation highlights failures APIM absorbed, failover depth, and caller-visible outcomes.',
            bold_tokens=(
                'The green checkmark indicates that every request returned HTTP 200',
                'The amber warning triangle indicates that one or more requests returned a caller-visible non-200 response',
            ),
        ),
        column_widths=['4%', '5%', '10%', '6%', '6%', '5%', '11%', '17%', '36%'],
    )
for test_id, title, description, api_results, _, backend_labels in scenario_definitions:
    add_scenario_figure(report, f'{test_id}) {title}', description, api_results, backend_labels)

if 'distribution_frame' in locals():
    distribution_frame = inference_failover_helpers.with_backend_identifier(distribution_frame)
    report.add_table(
        'Gateway-Recorded Backend Distribution',
        distribution_frame.columns.tolist(),
        distribution_frame.values.tolist(),
        'Log Analytics rows summarize the destination selected for each gateway request.',
    )
    distribution_figure, distribution_axis = plt.subplots(figsize=(12, 5))
    distribution_frame.pivot(index='API', columns='Backend', values='Requests').fillna(0).plot(kind='bar', ax=distribution_axis)
    distribution_axis.set_title('Gateway-recorded request distribution by backend')
    distribution_axis.set_xlabel('API')
    distribution_axis.set_ylabel('Requests')
    distribution_figure.subplots_adjust(bottom=0.4)
    report.add_figure('Gateway-Recorded Request Distribution', distribution_figure)
    plt.close(distribution_figure)
else:
    print_warning('Gateway distribution telemetry is not available for the HTML report yet')

if 'token_frame' in locals():
    report.add_table(
        'LLM Token Volume',
        token_frame.columns.tolist(),
        token_frame.values.tolist(),
        'Token totals confirm that LLM diagnostics captured successful inference traffic.',
    )
    token_figure, token_axis = plt.subplots(figsize=(8, 4))
    token_frame.groupby('API')['TotalTokens'].sum().plot(kind='bar', ax=token_axis)
    token_axis.set_title('LLM token volume by inference API')
    token_axis.set_xlabel('Inference API')
    token_axis.set_ylabel('Total tokens')
    token_axis.tick_params(axis='x', rotation=0)
    token_figure.tight_layout()
    report.add_figure('LLM Token Volume By Inference API', token_figure)
    plt.close(token_figure)
else:
    print_warning('Token telemetry is not available for the HTML report yet')

report.add_links(
    'Azure Views',
    {
        'Workbook': f'{portal_base}{workbook_id}/workbook',
        'Log Analytics': f'{portal_base}{log_analytics_id}/logs',
        'API Management': f'{portal_base}{apim_id}/overview',
    },
)
report_path = report.write(Path(tempfile.gettempdir()) / 'apim-samples-reports' / sample_folder / 'inference-failover-report.html')
report_url = report_path.as_uri()
print_ok(f'Local HTML report ready: {report_url}')


## Open Azure Views

Print links to the deployed workbook, Log Analytics workspace, and APIM instance for deeper exploration after the notebook graphs complete.

In [ ]:
from console import print_ok, print_val, print_warning

portal_base = f'https://portal.azure.com/#@{tenant_id}/resource'
apim_id = (
    f'/subscriptions/{subscription_id}/resourceGroups/{rg_name}'
    f'/providers/Microsoft.ApiManagement/service/{apim_name}'
)
print_val('Workbook', f'{portal_base}{workbook_id}/workbook')
print_val('Log Analytics', f'{portal_base}{log_analytics_id}/logs')
print_val('API Management', f'{portal_base}{apim_id}/overview')
if enable_event_hub_export:
    print_val('Event Hubs namespace', f'{portal_base}{event_hub_namespace_id}/overview')
if 'report_url' in locals():
    print_val('Local HTML report', report_url)
else:
    print_warning('Run the local HTML report cell to generate a shareable summary')
print_ok('All done!')